# Notebook 06a — Front Visibility Ground Truth

**Objective:** manually define when the single fish is observable in the raw Front video, expand interval labels to frames, and reinterpret the accepted `CONF068_N1` detections against `expected_visible_count`.

This notebook does not infer visibility, create bounding-box ground truth, rerun detection, sweep confidence, track fish, or start Notebook 07. `IN_SHELTER` means the expected visible count is zero and is not a detector miss.

## 1. Experiment metadata and CONFIG

In [5]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, subprocess, sys
import cv2
import numpy as np
import pandas as pd
import torch
import yaml

STARTED_AT = datetime.now(timezone.utc).isoformat()
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
CONDA_ENV = os.environ.get('CONDA_DEFAULT_ENV', '')
GIT_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, capture_output=True, text=True, check=True).stdout.strip()
CUDA_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else 'NOT_AVAILABLE'

TOTAL_FISH_COUNT = 1
DETECTION_CONF = 0.68
NMS_IOU = 0.70
IMGSZ = 640
CONF_TAG = f'conf{int(round(DETECTION_CONF * 100)):03d}'
FISH_COUNT_TAG = f'n{TOTAL_FISH_COUNT}'
RUN_TAG = f'{CONF_TAG}_{FISH_COUNT_TAG}'
SOURCE_EXPERIMENT_ID = f'FRONT_VIDEO_DET_{CONF_TAG.upper()}_{FISH_COUNT_TAG.upper()}_001'
EXPERIMENT_ID = f'FRONT_VISIBILITY_AUDIT_{CONF_TAG.upper()}_{FISH_COUNT_TAG.upper()}_001'
EXPECTED_MODEL_SHA256 = '750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'detection'
SOURCE_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'front' / 'detection' / RUN_TAG
SOURCE_LOG_DIR = PROJECT_ROOT / 'logs' / 'detection' / SOURCE_EXPERIMENT_ID
FRAME_COUNTS_PATH = SOURCE_OUTPUT_DIR / 'frame_counts.csv'
DETECTIONS_PATH = SOURCE_OUTPUT_DIR / 'detections.csv'
SOURCE_CONFIG_PATH = SOURCE_LOG_DIR / 'config.yaml'
SOURCE_SUMMARY_PATH = SOURCE_LOG_DIR / 'summary.json'
INTERVALS_PATH = RESULTS_DIR / 'front_visibility_intervals.csv'
SHORT_ZERO_REVIEW_PATH = RESULTS_DIR / f'front_short_zero_run_review_{RUN_TAG}.csv'
FRAME_LABELS_PATH = RESULTS_DIR / 'front_visibility_frame_labels.csv'
VISIBILITY_METRICS_PATH = RESULTS_DIR / f'front_detection_visibility_aware_{RUN_TAG}.csv'
VISIBLE_MISS_RUNS_PATH = RESULTS_DIR / f'front_visible_miss_runs_{RUN_TAG}.csv'
REVIEW_ROOT = PROJECT_ROOT / 'outputs' / 'front' / 'detection' / 'visibility_review' / RUN_TAG
VISIBLE_MISS_REVIEW_DIR = REVIEW_ROOT / 'visible_miss_runs'
SHELTER_REVIEW_DIR = REVIEW_ROOT / 'shelter'
SHORT_ZERO_REVIEW_DIR = REVIEW_ROOT / 'short_zero_runs'
SHORT_RUN_MAX_FRAMES = 2
CONTEXT_FRAMES = 3
MAX_SHELTER_REVIEW_IMAGES = 30
LOG_DIR = PROJECT_ROOT / 'logs' / 'detection' / EXPERIMENT_ID
ALLOWED_STATES = {'VISIBLE', 'IN_SHELTER', 'PARTIALLY_VISIBLE', 'UNCERTAIN'}
INTERVAL_COLUMNS = ['start_frame', 'end_frame', 'start_time_sec', 'end_time_sec', 'visibility_state', 'review_note']

CONFIG = {'experiment_id': EXPERIMENT_ID, 'source_experiment_id': SOURCE_EXPERIMENT_ID, 'total_fish_count': TOTAL_FISH_COUNT, 'detection_conf': DETECTION_CONF, 'nms_iou': NMS_IOU, 'imgsz': IMGSZ, 'run_tag': RUN_TAG, 'intervals_path': str(INTERVALS_PATH.relative_to(PROJECT_ROOT)), 'short_zero_review_path': str(SHORT_ZERO_REVIEW_PATH.relative_to(PROJECT_ROOT)), 'frame_labels_path': str(FRAME_LABELS_PATH.relative_to(PROJECT_ROOT)), 'visibility_metrics_path': str(VISIBILITY_METRICS_PATH.relative_to(PROJECT_ROOT)), 'visible_miss_runs_path': str(VISIBLE_MISS_RUNS_PATH.relative_to(PROJECT_ROOT)), 'review_root': str(REVIEW_ROOT.relative_to(PROJECT_ROOT)), 'short_run_max_frames': SHORT_RUN_MAX_FRAMES, 'context_frames': CONTEXT_FRAMES, 'max_shelter_review_images': MAX_SHELTER_REVIEW_IMAGES}
print(f'experiment_id: {EXPERIMENT_ID}')
print(f'datetime_utc: {STARTED_AT}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')
print(f'Conda environment: {CONDA_ENV}')
print(f'Device: CUDA available={CUDA_AVAILABLE}; GPU={GPU_NAME}')
print(f'Git commit: {GIT_COMMIT}')
print('CONFIG — FRONT VISIBILITY GROUND TRUTH')
for key, value in CONFIG.items(): print(f'{key}: {value}')
print('Visibility is assigned manually by USER; no AI visibility inference is performed.')

experiment_id: FRONT_VISIBILITY_AUDIT_CONF068_N1_001
datetime_utc: 2026-08-17T10:11:43.768547+00:00
PROJECT_ROOT: /home/diy-hus/fish
Python executable: /home/diy-hus/miniconda3/envs/fish/bin/python
Conda environment: fish
Device: CUDA available=True; GPU=NVIDIA GeForce RTX 3050
Git commit: 4cfac689c08d0f3f25bdee9cb8aac99d3202b9ee
CONFIG — FRONT VISIBILITY GROUND TRUTH
experiment_id: FRONT_VISIBILITY_AUDIT_CONF068_N1_001
source_experiment_id: FRONT_VIDEO_DET_CONF068_N1_001
total_fish_count: 1
detection_conf: 0.68
nms_iou: 0.7
imgsz: 640
run_tag: conf068_n1
intervals_path: results/detection/front_visibility_intervals.csv
short_zero_review_path: results/detection/front_short_zero_run_review_conf068_n1.csv
frame_labels_path: results/detection/front_visibility_frame_labels.csv
visibility_metrics_path: results/detection/front_detection_visibility_aware_conf068_n1.csv
visible_miss_runs_path: results/detection/front_visible_miss_runs_conf068_n1.csv
review_root: outputs/front/detection/visibili

## 2. Validate the accepted detection source and provenance

In [6]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''): digest.update(chunk)
    return digest.hexdigest()

assert CONDA_ENV == 'fish', f'FAIL preflight: expected Conda env fish, found {CONDA_ENV!r}'
for required_path in (FRAME_COUNTS_PATH, SOURCE_CONFIG_PATH, SOURCE_SUMMARY_PATH):
    assert required_path.is_file(), f'FAIL preflight: missing accepted Notebook 05 evidence: {required_path.relative_to(PROJECT_ROOT)}'
SOURCE_CONFIG = yaml.safe_load(SOURCE_CONFIG_PATH.read_text(encoding='utf-8'))
SOURCE_SUMMARY = json.loads(SOURCE_SUMMARY_PATH.read_text(encoding='utf-8'))
assert SOURCE_CONFIG.get('experiment_id') == SOURCE_EXPERIMENT_ID, 'FAIL provenance: source experiment ID mismatch.'
assert int(SOURCE_CONFIG.get('experimental_fish_count')) == TOTAL_FISH_COUNT, 'FAIL provenance: source fish count mismatch.'
assert np.isclose(float(SOURCE_CONFIG.get('detection_conf')), DETECTION_CONF), 'FAIL provenance: detection confidence mismatch.'
assert np.isclose(float(SOURCE_CONFIG.get('nms_iou')), NMS_IOU), 'FAIL provenance: NMS IoU mismatch.'
assert int(SOURCE_CONFIG.get('imgsz')) == IMGSZ, 'FAIL provenance: image size mismatch.'
assert SOURCE_SUMMARY.get('model_sha256') == EXPECTED_MODEL_SHA256, 'FAIL provenance: model SHA-256 mismatch.'
VIDEO_PATH = PROJECT_ROOT / SOURCE_CONFIG['video_path']
assert VIDEO_PATH.is_file(), f'FAIL preflight: source video missing: {VIDEO_PATH}'
VIDEO_SHA256 = sha256_file(VIDEO_PATH)
assert VIDEO_SHA256 == SOURCE_CONFIG.get('video_sha256') == SOURCE_SUMMARY.get('video_sha256'), 'FAIL provenance: video SHA-256 mismatch.'
VIDEO_FPS = float(SOURCE_CONFIG['video_fps'])
TOTAL_FRAMES = int(SOURCE_CONFIG['video_frame_count'])
assert VIDEO_FPS > 0 and TOTAL_FRAMES > 0, 'FAIL preflight: invalid video metadata.'
FRAME_COUNTS = pd.read_csv(FRAME_COUNTS_PATH)
required_count_columns = {'frame_index', 'time_sec', 'n_detections'}
assert required_count_columns.issubset(FRAME_COUNTS.columns), f'FAIL source schema: missing {required_count_columns - set(FRAME_COUNTS.columns)}'
FRAME_COUNTS = FRAME_COUNTS.sort_values('frame_index').reset_index(drop=True)
assert len(FRAME_COUNTS) == TOTAL_FRAMES, 'FAIL source completeness: frame count row total differs from video metadata.'
assert np.array_equal(FRAME_COUNTS['frame_index'].to_numpy(), np.arange(TOTAL_FRAMES)), 'FAIL source completeness: frame indices are not contiguous.'
assert (FRAME_COUNTS['n_detections'] >= 0).all(), 'FAIL source schema: negative detection count.'
print(f'Source: {SOURCE_EXPERIMENT_ID}')
print(f'Video: {VIDEO_PATH.relative_to(PROJECT_ROOT)}; SHA-256={VIDEO_SHA256}')
print(f'Frames: {TOTAL_FRAMES}; FPS: {VIDEO_FPS:.6f}; duration: {TOTAL_FRAMES / VIDEO_FPS:.3f} sec')
print('SOURCE_PROVENANCE: PASS')

Source: FRONT_VIDEO_DET_CONF068_N1_001
Video: data/raw/front/4.mp4; SHA-256=3f8587344beba8dcc6f835d1ed94aa8daadb06a369b5b96977ee902f035b5700
Frames: 3431; FPS: 28.668432; duration: 119.679 sec
SOURCE_PROVENANCE: PASS


## 3. Automatically discover and prepare short zero-run review

The notebook finds every contiguous `n_detections == 0` run no longer than `SHORT_RUN_MAX_FRAMES`. Runs already fully covered by a confirmed `IN_SHELTER` interval do not require short-run review. For every remaining run, it exports one context contact sheet. Only USER may replace `UNREVIEWED` in the review CSV with an allowed visibility label. Existing USER labels and notes are preserved by the stable `(start_frame, end_frame)` key.

In [7]:
REVIEW_COLUMNS = ['run_id', 'start_frame', 'end_frame', 'duration_frames', 'start_time_sec', 'end_time_sec', 'interval_visibility_state', 'contact_sheet_path', 'manual_visibility_label', 'manual_note']
MANUAL_REVIEW_STATES = ALLOWED_STATES | {'UNREVIEWED'}
INTERVALS_PREVIEW = pd.DataFrame(columns=INTERVAL_COLUMNS)
if INTERVALS_PATH.is_file() and INTERVALS_PATH.stat().st_size > 0:
    try:
        INTERVALS_PREVIEW = pd.read_csv(INTERVALS_PATH)
    except pd.errors.EmptyDataError:
        INTERVALS_PREVIEW = pd.DataFrame(columns=INTERVAL_COLUMNS)
if not INTERVALS_PREVIEW.empty or len(INTERVALS_PREVIEW.columns):
    assert list(INTERVALS_PREVIEW.columns) == INTERVAL_COLUMNS, f'FAIL interval schema: expected columns {INTERVAL_COLUMNS}'
if not INTERVALS_PREVIEW.empty:
    for column in ('start_frame', 'end_frame'):
        INTERVALS_PREVIEW[column] = pd.to_numeric(INTERVALS_PREVIEW[column], errors='raise').astype(int)
    INTERVALS_PREVIEW['visibility_state'] = INTERVALS_PREVIEW['visibility_state'].astype(str).str.strip().str.upper()
    assert set(INTERVALS_PREVIEW['visibility_state']).issubset(ALLOWED_STATES), 'FAIL interval preview: invalid visibility state.'

zero_indices = FRAME_COUNTS.loc[FRAME_COUNTS['n_detections'] == 0, 'frame_index'].to_numpy(dtype=int)
all_zero_runs = []
if len(zero_indices):
    run_start = run_end = int(zero_indices[0])
    for frame_index in zero_indices[1:]:
        frame_index = int(frame_index)
        if frame_index == run_end + 1:
            run_end = frame_index
        else:
            all_zero_runs.append((run_start, run_end))
            run_start = run_end = frame_index
    all_zero_runs.append((run_start, run_end))
short_zero_runs = [(start, end) for start, end in all_zero_runs if end - start + 1 <= SHORT_RUN_MAX_FRAMES]

def interval_state_for_run(start_frame, end_frame):
    covering = INTERVALS_PREVIEW[(INTERVALS_PREVIEW['start_frame'] <= start_frame) & (INTERVALS_PREVIEW['end_frame'] >= end_frame)]
    if len(covering) == 1: return str(covering.iloc[0]['visibility_state'])
    overlapping = INTERVALS_PREVIEW[(INTERVALS_PREVIEW['start_frame'] <= end_frame) & (INTERVALS_PREVIEW['end_frame'] >= start_frame)]
    return 'UNASSIGNED' if overlapping.empty else 'MIXED'

short_run_rows = []
for discovered_run_id, (start, end) in enumerate(short_zero_runs, start=1):
    interval_state = interval_state_for_run(start, end)
    short_run_rows.append({'run_id': discovered_run_id, 'start_frame': start, 'end_frame': end, 'duration_frames': end - start + 1, 'start_time_sec': start / VIDEO_FPS, 'end_time_sec': end / VIDEO_FPS, 'interval_visibility_state': interval_state})
SHORT_ZERO_RUNS_DF = pd.DataFrame(short_run_rows)
SHORT_ZERO_REVIEW_DIR.mkdir(parents=True, exist_ok=True)

contact_sheet_paths = {}
video_capture = cv2.VideoCapture(str(VIDEO_PATH))
if not video_capture.isOpened(): raise RuntimeError(f'FAIL short-run review: cannot open {VIDEO_PATH}')
try:
    for run in SHORT_ZERO_RUNS_DF.itertuples(index=False):
        if run.interval_visibility_state == 'IN_SHELTER': continue
        context_start = max(0, run.start_frame - CONTEXT_FRAMES)
        context_end = min(TOTAL_FRAMES - 1, run.end_frame + CONTEXT_FRAMES)
        tiles = []
        for frame_index in range(context_start, context_end + 1):
            video_capture.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
            ok, frame = video_capture.read()
            if not ok or frame is None: raise RuntimeError(f'FAIL contact sheet: cannot decode frame {frame_index}')
            tile_width = 320
            tile_height = max(1, int(round(frame.shape[0] * tile_width / frame.shape[1])))
            tile = cv2.resize(frame, (tile_width, tile_height))
            is_zero_run_frame = run.start_frame <= frame_index <= run.end_frame
            border_color = (0, 0, 255) if is_zero_run_frame else (180, 180, 180)
            cv2.rectangle(tile, (0, 0), (tile_width - 1, tile_height - 1), border_color, 5 if is_zero_run_frame else 2)
            detected_count = int(FRAME_COUNTS.loc[FRAME_COUNTS['frame_index'] == frame_index, 'n_detections'].iloc[0])
            cv2.rectangle(tile, (0, 0), (tile_width, 28), (0, 0, 0), -1)
            cv2.putText(tile, f'frame={frame_index} t={frame_index / VIDEO_FPS:.2f}s det={detected_count}', (6, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255, 255, 255), 1, cv2.LINE_AA)
            tiles.append(tile)
        grid_columns = 4
        grid_rows = int(np.ceil(len(tiles) / grid_columns))
        sheet = np.zeros((grid_rows * tiles[0].shape[0], grid_columns * tiles[0].shape[1], 3), dtype=np.uint8)
        for tile_index, tile in enumerate(tiles):
            row, column = divmod(tile_index, grid_columns)
            sheet[row * tile.shape[0]:(row + 1) * tile.shape[0], column * tile.shape[1]:(column + 1) * tile.shape[1]] = tile
        contact_sheet_path = SHORT_ZERO_REVIEW_DIR / f'run_{run.run_id:03d}_frames_{run.start_frame:06d}_{run.end_frame:06d}.jpg'
        if not cv2.imwrite(str(contact_sheet_path), sheet): raise RuntimeError(f'FAIL contact sheet: cannot write {contact_sheet_path}')
        contact_sheet_paths[(run.start_frame, run.end_frame)] = str(contact_sheet_path.relative_to(PROJECT_ROOT))
finally:
    video_capture.release()

existing_reviews = {}
EXISTING_REVIEW_DF = pd.DataFrame(columns=REVIEW_COLUMNS)
if SHORT_ZERO_REVIEW_PATH.is_file() and SHORT_ZERO_REVIEW_PATH.stat().st_size > 0:
    try:
        EXISTING_REVIEW_DF = pd.read_csv(SHORT_ZERO_REVIEW_PATH)
    except pd.errors.EmptyDataError:
        EXISTING_REVIEW_DF = pd.DataFrame(columns=REVIEW_COLUMNS)
if not EXISTING_REVIEW_DF.empty:
    EXISTING_REVIEW_DF = EXISTING_REVIEW_DF.fillna({'manual_visibility_label': 'UNREVIEWED', 'manual_note': ''})
    assert set(REVIEW_COLUMNS).issubset(EXISTING_REVIEW_DF.columns), 'FAIL review CSV schema: required columns are missing.'
    EXISTING_REVIEW_DF['manual_visibility_label'] = EXISTING_REVIEW_DF['manual_visibility_label'].astype(str).str.strip().str.upper()
    invalid_review_labels = sorted(set(EXISTING_REVIEW_DF['manual_visibility_label']) - MANUAL_REVIEW_STATES)
    assert not invalid_review_labels, f'FAIL review CSV: invalid manual labels {invalid_review_labels}'
    assert not EXISTING_REVIEW_DF.duplicated(['start_frame', 'end_frame']).any(), 'FAIL review CSV: duplicate stable keys.'
    existing_reviews = {(int(row.start_frame), int(row.end_frame)): (row.manual_visibility_label, str(row.manual_note)) for row in EXISTING_REVIEW_DF.itertuples(index=False)}
review_output_rows = []
for run in SHORT_ZERO_RUNS_DF.itertuples(index=False):
    stable_key = (run.start_frame, run.end_frame)
    manual_label, manual_note = existing_reviews.get(stable_key, ('UNREVIEWED', ''))
    review_output_rows.append({**run._asdict(), 'contact_sheet_path': contact_sheet_paths.get(stable_key, ''), 'manual_visibility_label': manual_label, 'manual_note': manual_note})
SHORT_ZERO_REVIEW_DF = pd.DataFrame(review_output_rows, columns=REVIEW_COLUMNS)
SHORT_ZERO_REVIEW_DF.to_csv(SHORT_ZERO_REVIEW_PATH, index=False)
SHORT_ZERO_REVIEW_REQUIRED_DF = SHORT_ZERO_REVIEW_DF[SHORT_ZERO_REVIEW_DF['interval_visibility_state'] != 'IN_SHELTER'].copy()
SHORT_ZERO_RUNS_TOTAL = len(SHORT_ZERO_REVIEW_REQUIRED_DF)
SHORT_ZERO_RUNS_REVIEWED = int(SHORT_ZERO_REVIEW_REQUIRED_DF['manual_visibility_label'].isin(ALLOWED_STATES).sum())
SHORT_ZERO_RUNS_UNREVIEWED = int((SHORT_ZERO_REVIEW_REQUIRED_DF['manual_visibility_label'] == 'UNREVIEWED').sum())
print(f'Short zero runs requiring review: {SHORT_ZERO_RUNS_TOTAL}')
print(f'Reviewed: {SHORT_ZERO_RUNS_REVIEWED}; unreviewed: {SHORT_ZERO_RUNS_UNREVIEWED}')
print(f'Created/updated {SHORT_ZERO_REVIEW_PATH.relative_to(PROJECT_ROOT)} without overwriting existing USER labels or notes.')
display(SHORT_ZERO_REVIEW_DF)

Short zero runs requiring review: 24
Reviewed: 24; unreviewed: 0
Created/updated results/detection/front_short_zero_run_review_conf068_n1.csv without overwriting existing USER labels or notes.


,run_id,start_frame,end_frame,duration_frames,start_time_sec,end_time_sec,interval_visibility_state,contact_sheet_path,manual_visibility_label,manual_note
0,1,381,382,2,13.289879,13.324761,UNASSIGNED,outputs/front/detection/visibility_review/conf...,VISIBLE,
1,2,387,388,2,13.499169,13.534050,UNASSIGNED,outputs/front/detection/visibility_review/conf...,VISIBLE,
2,3,401,401,1,13.987511,13.987511,UNASSIGNED,outputs/front/detection/visibility_review/conf...,VISIBLE,
3,4,410,410,1,14.301445,14.301445,UNASSIGNED,outputs/front/detection/visibility_review/conf...,VISIBLE,
4,5,412,413,2,14.371208,14.406090,UNASSIGNED,outputs/front/detection/visibility_review/conf...,VISIBLE,
5,6,415,415,1,14.475853,14.475853,UNASSIGNED,outputs/front/detection/visibility_review/conf...,VISIBLE,
6,7,788,788,1,27.486679,27.486679,UNASSIGNED,outputs/front/detection/visibility_review/conf...,VISIBLE,
7,8,918,919,2,32.021284,32.056166,UNASSIGNED,outputs/front/detection/visibility_review/conf...,PARTIALLY_VISIBLE,
8,9,925,926,2,32.265455,32.300337,UNASSIGNED,outputs/front/detection/visibility_review/conf...,PARTIALLY_VISIBLE,
9,10,2179,2179,1,76.006947,76.006947,UNASSIGNED,outputs/front/detection/visibility_review/conf...,PARTIALLY_VISIBLE,


## 4. Validate the USER manual visibility interval baseline

Allowed states are `VISIBLE`, `IN_SHELTER`, `PARTIALLY_VISIBLE`, and `UNCERTAIN`. Review the raw video and edit `results/detection/front_visibility_intervals.csv`. Intervals are inclusive, must cover frames `0` through `TOTAL_FRAMES - 1` without overlap or unexplained gaps, and must not be inferred automatically. Time fields may be left blank; the notebook derives them from frame indices and source FPS.

In [8]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
INTERVALS = pd.DataFrame(columns=INTERVAL_COLUMNS)
if INTERVALS_PATH.is_file() and INTERVALS_PATH.stat().st_size > 0:
    try:
        INTERVALS = pd.read_csv(INTERVALS_PATH)
    except pd.errors.EmptyDataError:
        INTERVALS = pd.DataFrame(columns=INTERVAL_COLUMNS)
assert list(INTERVALS.columns) == INTERVAL_COLUMNS, f'FAIL interval schema: expected columns {INTERVAL_COLUMNS}'
if INTERVALS.empty:
    raise RuntimeError(f'USER_ACTION_REQUIRED: {INTERVALS_PATH.relative_to(PROJECT_ROOT)} is empty. Enter manual visibility intervals and Run All again.')
INTERVALS['visibility_state'] = INTERVALS['visibility_state'].astype(str).str.strip().str.upper()
invalid_states = sorted(set(INTERVALS['visibility_state']) - ALLOWED_STATES)
assert not invalid_states, f'FAIL interval states: unsupported values {invalid_states}'
for column in ('start_frame', 'end_frame'):
    numeric = pd.to_numeric(INTERVALS[column], errors='coerce')
    assert numeric.notna().all(), f'FAIL intervals: {column} contains missing/non-numeric values.'
    assert np.allclose(numeric, np.round(numeric)), f'FAIL intervals: {column} must contain integers.'
    INTERVALS[column] = numeric.astype(int)
INTERVALS = INTERVALS.sort_values(['start_frame', 'end_frame']).reset_index(drop=True)
assert (INTERVALS['start_frame'] <= INTERVALS['end_frame']).all(), 'FAIL intervals: start_frame exceeds end_frame.'
assert INTERVALS.iloc[0]['start_frame'] == 0, 'FAIL intervals: coverage must begin at frame 0.'
assert INTERVALS.iloc[-1]['end_frame'] == TOTAL_FRAMES - 1, f'FAIL intervals: coverage must end at frame {TOTAL_FRAMES - 1}.'
assert INTERVALS['start_frame'].min() >= 0 and INTERVALS['end_frame'].max() < TOTAL_FRAMES, 'FAIL intervals: frame range is outside the video.'
for previous, current in zip(INTERVALS.itertuples(index=False), INTERVALS.iloc[1:].itertuples(index=False)):
    assert current.start_frame == previous.end_frame + 1, f'FAIL intervals: overlap or unexplained gap between frames {previous.end_frame} and {current.start_frame}.'
derived_start = INTERVALS['start_frame'] / VIDEO_FPS
derived_end = INTERVALS['end_frame'] / VIDEO_FPS
for column, derived in (('start_time_sec', derived_start), ('end_time_sec', derived_end)):
    supplied = pd.to_numeric(INTERVALS[column], errors='coerce')
    mismatch = supplied.notna() & ~np.isclose(supplied.fillna(derived), derived, atol=0.5 / VIDEO_FPS)
    assert not mismatch.any(), f'FAIL intervals: {column} does not match frame index/source FPS within half a frame.'
    INTERVALS[column] = derived
INTERVALS['review_note'] = INTERVALS['review_note'].fillna('').astype(str)
print(f'Manual intervals validated read-only: {len(INTERVALS)}; complete coverage with no overlap or gap.')
display(INTERVALS)

RuntimeError: USER_ACTION_REQUIRED: results/detection/front_visibility_intervals.csv is empty. Enter manual visibility intervals and Run All again.

## 5. Create derived frame-level ground truth

In [ ]:
EXPECTED_VISIBLE_COUNT = {'VISIBLE': float(TOTAL_FISH_COUNT), 'IN_SHELTER': 0.0, 'PARTIALLY_VISIBLE': np.nan, 'UNCERTAIN': np.nan}
frame_label_parts = []
for interval in INTERVALS.itertuples(index=False):
    frame_indices = np.arange(interval.start_frame, interval.end_frame + 1, dtype=int)
    frame_label_parts.append(pd.DataFrame({'frame_index': frame_indices, 'time_sec': frame_indices / VIDEO_FPS, 'visibility_state': interval.visibility_state, 'expected_visible_count': EXPECTED_VISIBLE_COUNT[interval.visibility_state], 'visibility_source': 'MANUAL_INTERVAL'}))
FRAME_LABELS = pd.concat(frame_label_parts, ignore_index=True)
assert len(FRAME_LABELS) == TOTAL_FRAMES, 'FAIL expansion: frame label count differs from total frames.'
assert FRAME_LABELS['frame_index'].is_unique, 'FAIL expansion: duplicate frame labels.'
assert np.array_equal(FRAME_LABELS['frame_index'].to_numpy(), np.arange(TOTAL_FRAMES)), 'FAIL expansion: labels do not cover every frame in order.'
reviewed_short_runs = SHORT_ZERO_REVIEW_DF[SHORT_ZERO_REVIEW_DF['manual_visibility_label'].isin(ALLOWED_STATES)]
for review in reviewed_short_runs.itertuples(index=False):
    override_mask = FRAME_LABELS['frame_index'].between(review.start_frame, review.end_frame)
    assert int(override_mask.sum()) == review.duration_frames, f'FAIL short-run override coverage: frames {review.start_frame}-{review.end_frame}'
    FRAME_LABELS.loc[override_mask, 'visibility_state'] = review.manual_visibility_label
    FRAME_LABELS.loc[override_mask, 'expected_visible_count'] = EXPECTED_VISIBLE_COUNT[review.manual_visibility_label]
    FRAME_LABELS.loc[override_mask, 'visibility_source'] = 'MANUAL_SHORT_RUN_REVIEW'
assert set(FRAME_LABELS['visibility_source']).issubset({'MANUAL_INTERVAL', 'MANUAL_SHORT_RUN_REVIEW'}), 'FAIL derived provenance: invalid visibility source.'
FRAME_LABELS.to_csv(FRAME_LABELS_PATH, index=False)
print(f'Created {FRAME_LABELS_PATH.relative_to(PROJECT_ROOT)} ({len(FRAME_LABELS)} frame labels; {FRAME_LABELS_PATH.stat().st_size} bytes)')
display(FRAME_LABELS['visibility_state'].value_counts().rename_axis('visibility_state').reset_index(name='frames'))

## 6. Visibility-aware strict detection evaluation

In [ ]:
AUDIT = FRAME_LABELS.merge(FRAME_COUNTS[['frame_index', 'n_detections']], on='frame_index', how='left', validate='one_to_one')
assert AUDIT['n_detections'].notna().all(), 'FAIL merge: detections are missing for labeled frames.'
AUDIT['n_detections'] = AUDIT['n_detections'].astype(int)
VISIBLE = AUDIT[AUDIT['visibility_state'] == 'VISIBLE'].copy()
SHELTER = AUDIT[AUDIT['visibility_state'] == 'IN_SHELTER'].copy()
PARTIALLY_VISIBLE_FRAMES = int((AUDIT['visibility_state'] == 'PARTIALLY_VISIBLE').sum())
UNCERTAIN_FRAMES = int((AUDIT['visibility_state'] == 'UNCERTAIN').sum())
VISIBLE_FRAMES = len(VISIBLE); IN_SHELTER_FRAMES = len(SHELTER)
def safe_rate(mask, denominator):
    return float(np.sum(mask) / denominator) if denominator else None
VISIBLE_CORRECT_FRAMES = int((VISIBLE['n_detections'] == TOTAL_FISH_COUNT).sum())
VISIBLE_MISS_FRAMES = int((VISIBLE['n_detections'] == 0).sum())
VISIBLE_OVERCOUNT_FRAMES = int((VISIBLE['n_detections'] > TOTAL_FISH_COUNT).sum())
SHELTER_CORRECT_ZERO_FRAMES = int((SHELTER['n_detections'] == 0).sum())
SHELTER_UNEXPECTED_DETECTION_FRAMES = int((SHELTER['n_detections'] >= TOTAL_FISH_COUNT).sum())
VISIBLE_DETECTION_RATE = safe_rate(VISIBLE['n_detections'] == TOTAL_FISH_COUNT, VISIBLE_FRAMES)
VISIBLE_MISS_RATE = safe_rate(VISIBLE['n_detections'] == 0, VISIBLE_FRAMES)
VISIBLE_OVERCOUNT_RATE = safe_rate(VISIBLE['n_detections'] > TOTAL_FISH_COUNT, VISIBLE_FRAMES)
SHELTER_CORRECT_ZERO_RATE = safe_rate(SHELTER['n_detections'] == 0, IN_SHELTER_FRAMES)
SHELTER_UNEXPECTED_DETECTION_RATE = safe_rate(SHELTER['n_detections'] >= TOTAL_FISH_COUNT, IN_SHELTER_FRAMES)
visible_miss_frames_df = VISIBLE.loc[VISIBLE['n_detections'] == 0].copy()
visible_correct_frames_df = VISIBLE.loc[VISIBLE['n_detections'] == TOTAL_FISH_COUNT].copy()
visible_overcount_frames_df = VISIBLE.loc[VISIBLE['n_detections'] > TOTAL_FISH_COUNT].copy()
shelter_frames_df = SHELTER.copy()
unexpected_shelter_frames_df = SHELTER.loc[SHELTER['n_detections'] >= TOTAL_FISH_COUNT].copy()
print('STRICT EVALUATION — VISIBLE AND IN_SHELTER FRAMES ONLY')
print({'visible_frames': VISIBLE_FRAMES, 'visible_correct_frames': VISIBLE_CORRECT_FRAMES, 'visible_miss_frames': VISIBLE_MISS_FRAMES, 'visible_overcount_frames': VISIBLE_OVERCOUNT_FRAMES, 'visible_detection_rate': VISIBLE_DETECTION_RATE, 'visible_miss_rate': VISIBLE_MISS_RATE, 'visible_overcount_rate': VISIBLE_OVERCOUNT_RATE})
print({'in_shelter_frames': IN_SHELTER_FRAMES, 'shelter_correct_zero_frames': SHELTER_CORRECT_ZERO_FRAMES, 'shelter_unexpected_detection_frames': SHELTER_UNEXPECTED_DETECTION_FRAMES, 'shelter_correct_zero_rate': SHELTER_CORRECT_ZERO_RATE, 'shelter_unexpected_detection_rate': SHELTER_UNEXPECTED_DETECTION_RATE})
print({'partially_visible_frames_excluded': PARTIALLY_VISIBLE_FRAMES, 'uncertain_frames_excluded': UNCERTAIN_FRAMES})

## 7. Temporal missing-detection runs within VISIBLE periods only

In [ ]:
AUDIT['strict_visible_miss'] = (AUDIT['visibility_state'] == 'VISIBLE') & (AUDIT['n_detections'] == 0)
miss_indices = AUDIT.loc[AUDIT['strict_visible_miss'], 'frame_index'].to_numpy(dtype=int)
miss_runs = []
if len(miss_indices):
    run_start = run_end = int(miss_indices[0])
    for frame_index in miss_indices[1:]:
        frame_index = int(frame_index)
        if frame_index == run_end + 1:
            run_end = frame_index
        else:
            miss_runs.append((run_start, run_end))
            run_start = run_end = frame_index
    miss_runs.append((run_start, run_end))
miss_run_rows = []
for run_id, (start, end) in enumerate(miss_runs, start=1):
    duration_frames = end - start + 1
    miss_run_rows.append({'run_id': run_id, 'start_frame': start, 'end_frame': end, 'start_time_sec': start / VIDEO_FPS, 'end_time_sec': end / VIDEO_FPS, 'duration_frames': duration_frames, 'duration_sec': duration_frames / VIDEO_FPS, 'middle_frame': (start + end) // 2})
VISIBLE_MISS_RUNS_DF = pd.DataFrame(miss_run_rows, columns=['run_id', 'start_frame', 'end_frame', 'start_time_sec', 'end_time_sec', 'duration_frames', 'duration_sec', 'middle_frame'])
VISIBLE_MISS_RUNS = len(VISIBLE_MISS_RUNS_DF)
LONGEST_VISIBLE_MISS_RUN_FRAMES = int(VISIBLE_MISS_RUNS_DF['duration_frames'].max()) if VISIBLE_MISS_RUNS else 0
LONGEST_VISIBLE_MISS_RUN_SEC = float(VISIBLE_MISS_RUNS_DF['duration_sec'].max()) if VISIBLE_MISS_RUNS else 0.0
MEDIAN_VISIBLE_MISS_RUN_FRAMES = float(VISIBLE_MISS_RUNS_DF['duration_frames'].median()) if VISIBLE_MISS_RUNS else 0.0
P95_VISIBLE_MISS_RUN_FRAMES = float(VISIBLE_MISS_RUNS_DF['duration_frames'].quantile(0.95)) if VISIBLE_MISS_RUNS else 0.0
VISIBLE_MISS_RUNS_DF.to_csv(VISIBLE_MISS_RUNS_PATH, index=False)
print(f'Visible miss runs: {VISIBLE_MISS_RUNS}')
print(f'Longest visible miss run: {LONGEST_VISIBLE_MISS_RUN_FRAMES} frames ({LONGEST_VISIBLE_MISS_RUN_SEC:.3f} sec)')
print(f'Median/P95 visible miss run: {MEDIAN_VISIBLE_MISS_RUN_FRAMES:.3f}/{P95_VISIBLE_MISS_RUN_FRAMES:.3f} frames')
print(f'Created {VISIBLE_MISS_RUNS_PATH.relative_to(PROJECT_ROOT)} ({VISIBLE_MISS_RUNS} runs)')
print('Runs cannot cross IN_SHELTER, PARTIALLY_VISIBLE, or UNCERTAIN frames.')
display(VISIBLE_MISS_RUNS_DF.sort_values('duration_frames', ascending=False).head(20) if VISIBLE_MISS_RUNS else VISIBLE_MISS_RUNS_DF)

## 8. Extract representative review frames

For every `VISIBLE MISS` run, extract unique start/middle/end frames. Also extract a bounded set of start/middle/end frames from `IN_SHELTER` intervals. These images support manual review only: a visible miss is a **suspected miss while the fish is visible**, not an official false negative without bounding-box ground truth. Real detection boxes are drawn when detection-level evidence exists; no box is synthesized.

In [ ]:
DETECTION_COLUMNS = ['frame_index', 'time_sec', 'class_id', 'confidence', 'x1', 'y1', 'x2', 'y2', 'cx', 'cy']
if DETECTIONS_PATH.is_file():
    DETECTIONS = pd.read_csv(DETECTIONS_PATH)
    assert set(DETECTION_COLUMNS).issubset(DETECTIONS.columns), 'FAIL detection-level schema: required bbox columns are missing.'
    assert DETECTIONS['frame_index'].between(0, TOTAL_FRAMES - 1).all(), 'FAIL detection-level evidence: frame index outside video.'
else:
    DETECTIONS = pd.DataFrame(columns=DETECTION_COLUMNS)
    print('WARNING: detections.csv is unavailable; review images will contain no bbox overlays.')

review_rows = []
for run in VISIBLE_MISS_RUNS_DF.itertuples(index=False):
    candidates = [('START', run.start_frame), ('MIDDLE', run.middle_frame), ('END', run.end_frame)]
    used_frames = set()
    for position, frame_index in candidates:
        if frame_index in used_frames: continue
        used_frames.add(frame_index)
        review_rows.append({'review_type': 'VISIBLE_MISS', 'group_id': int(run.run_id), 'frame_index': int(frame_index), 'position_in_run': position, 'output_dir': VISIBLE_MISS_REVIEW_DIR})
shelter_intervals = INTERVALS.loc[INTERVALS['visibility_state'] == 'IN_SHELTER'].reset_index(drop=True)
shelter_review_rows = []
for shelter_id, interval in enumerate(shelter_intervals.itertuples(index=False), start=1):
    candidates = [('START', interval.start_frame), ('MIDDLE', (interval.start_frame + interval.end_frame) // 2), ('END', interval.end_frame)]
    used_frames = set()
    for position, frame_index in candidates:
        if frame_index in used_frames: continue
        used_frames.add(frame_index)
        shelter_review_rows.append({'review_type': 'IN_SHELTER', 'group_id': shelter_id, 'frame_index': int(frame_index), 'position_in_run': position, 'output_dir': SHELTER_REVIEW_DIR})
review_rows.extend(shelter_review_rows[:MAX_SHELTER_REVIEW_IMAGES])
REVIEW_PLAN = pd.DataFrame(review_rows, columns=['review_type', 'group_id', 'frame_index', 'position_in_run', 'output_dir'])
assert not REVIEW_PLAN.duplicated(['review_type', 'frame_index']).any(), 'FAIL review plan: duplicate review frame within a review type.'
VISIBLE_MISS_REVIEW_DIR.mkdir(parents=True, exist_ok=True)
SHELTER_REVIEW_DIR.mkdir(parents=True, exist_ok=True)

video_capture = cv2.VideoCapture(str(VIDEO_PATH))
if not video_capture.isOpened(): raise RuntimeError(f'FAIL review extraction: cannot open {VIDEO_PATH}')
review_manifest_rows = []
try:
    for item in REVIEW_PLAN.itertuples(index=False):
        video_capture.set(cv2.CAP_PROP_POS_FRAMES, item.frame_index)
        ok, frame = video_capture.read()
        if not ok or frame is None: raise RuntimeError(f'FAIL review extraction: cannot decode frame {item.frame_index}')
        audit_row = AUDIT.loc[AUDIT['frame_index'] == item.frame_index].iloc[0]
        frame_detections = DETECTIONS.loc[DETECTIONS['frame_index'] == item.frame_index]
        for detection in frame_detections.itertuples(index=False):
            p1 = (int(round(detection.x1)), int(round(detection.y1)))
            p2 = (int(round(detection.x2)), int(round(detection.y2)))
            cv2.rectangle(frame, p1, p2, (0, 220, 0), 2)
            cv2.putText(frame, f'Ca {detection.confidence:.2f}', (p1[0], max(18, p1[1] - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 220, 0), 2, cv2.LINE_AA)
        expected_text = 'NaN' if pd.isna(audit_row.expected_visible_count) else str(int(audit_row.expected_visible_count))
        header_lines = [f'frame={item.frame_index} time={audit_row.time_sec:.3f}s state={audit_row.visibility_state}', f'detections={audit_row.n_detections} expected_visible={expected_text} run_id={item.group_id} position={item.position_in_run}']
        cv2.rectangle(frame, (0, 0), (min(frame.shape[1], 1100), 66), (0, 0, 0), -1)
        for line_index, text in enumerate(header_lines): cv2.putText(frame, text, (10, 25 + 30 * line_index), cv2.FONT_HERSHEY_SIMPLEX, 0.66, (255, 255, 255), 2, cv2.LINE_AA)
        prefix = f'run_{item.group_id:03d}' if item.review_type == 'VISIBLE_MISS' else f'shelter_interval_{item.group_id:03d}'
        filename = f'{prefix}_{item.position_in_run.lower()}_frame_{item.frame_index:06d}.jpg'
        output_path = item.output_dir / filename
        if not cv2.imwrite(str(output_path), frame): raise RuntimeError(f'FAIL review extraction: cannot write {output_path}')
        review_manifest_rows.append({'review_type': item.review_type, 'group_id': item.group_id, 'frame_index': item.frame_index, 'time_sec': float(audit_row.time_sec), 'visibility_state': audit_row.visibility_state, 'n_detections': int(audit_row.n_detections), 'expected_visible_count': audit_row.expected_visible_count, 'position_in_run': item.position_in_run, 'output_path': str(output_path.relative_to(PROJECT_ROOT))})
finally:
    video_capture.release()
REVIEW_MANIFEST = pd.DataFrame(review_manifest_rows)
REPRESENTATIVE_REVIEW_IMAGES_CREATED = len(REVIEW_MANIFEST)
SHORT_CONTACT_SHEETS_CREATED = len(contact_sheet_paths)
REVIEW_IMAGES_CREATED = REPRESENTATIVE_REVIEW_IMAGES_CREATED + SHORT_CONTACT_SHEETS_CREATED
assert REPRESENTATIVE_REVIEW_IMAGES_CREATED == len(REVIEW_PLAN), 'FAIL review extraction: not every planned review frame was written.'
print(f'Review images/contact sheets created: {REVIEW_IMAGES_CREATED}')
print(f'- short zero-run contact sheets: {SHORT_CONTACT_SHEETS_CREATED}')
print(f'- visible miss run images: {(REVIEW_MANIFEST.review_type == "VISIBLE_MISS").sum() if REPRESENTATIVE_REVIEW_IMAGES_CREATED else 0}')
print(f'- shelter images: {(REVIEW_MANIFEST.review_type == "IN_SHELTER").sum() if REPRESENTATIVE_REVIEW_IMAGES_CREATED else 0}')
display(REVIEW_MANIFEST.head(20))

## 9. Save compact evidence and reinterpret Notebook 06

In [ ]:
LOG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = LOG_DIR / 'config.yaml'
ENVIRONMENT_PATH = LOG_DIR / 'environment.txt'
SUMMARY_PATH = LOG_DIR / 'summary.json'
WARNING_NOTE = 'Previous N1 audit assumed expected_visible_count=1 for every frame. This assumption is invalid when the fish is inside the shelter. Visibility-aware metrics supersede the previous undercount metrics for detector performance interpretation.'
NOTES = [WARNING_NOTE, 'PARTIALLY_VISIBLE and UNCERTAIN frames are reported only; they are not misses, false detections, or part of strict evaluation.', 'VISIBLE MISS means suspected miss while the fish is visible; it is not an official false negative without bounding-box ground truth.']
WARNINGS = []
if PARTIALLY_VISIBLE_FRAMES or UNCERTAIN_FRAMES: WARNINGS.append('PARTIALLY_VISIBLE or UNCERTAIN frames remain and are excluded from strict evaluation.')
if SHORT_ZERO_RUNS_UNREVIEWED: WARNINGS.append(f'{SHORT_ZERO_RUNS_UNREVIEWED} short zero runs remain UNREVIEWED; open their contact sheets and edit only manual_visibility_label in the review CSV.')
CHECKPOINT_RESULT = 'PASS_WITH_WARNING' if WARNINGS else 'PASS'
METRICS_ROW = {'experiment_id': EXPERIMENT_ID, 'source_experiment_id': SOURCE_EXPERIMENT_ID, 'total_fish_count': TOTAL_FISH_COUNT, 'detection_conf': DETECTION_CONF, 'total_frames': TOTAL_FRAMES, 'short_zero_runs_total': SHORT_ZERO_RUNS_TOTAL, 'short_zero_runs_reviewed': SHORT_ZERO_RUNS_REVIEWED, 'short_zero_runs_unreviewed': SHORT_ZERO_RUNS_UNREVIEWED, 'visible_frames': VISIBLE_FRAMES, 'in_shelter_frames': IN_SHELTER_FRAMES, 'partially_visible_frames': PARTIALLY_VISIBLE_FRAMES, 'uncertain_frames': UNCERTAIN_FRAMES, 'visible_correct_frames': VISIBLE_CORRECT_FRAMES, 'visible_miss_frames': VISIBLE_MISS_FRAMES, 'visible_overcount_frames': VISIBLE_OVERCOUNT_FRAMES, 'visible_detection_rate': VISIBLE_DETECTION_RATE, 'visible_miss_rate': VISIBLE_MISS_RATE, 'visible_overcount_rate': VISIBLE_OVERCOUNT_RATE, 'shelter_correct_zero_frames': SHELTER_CORRECT_ZERO_FRAMES, 'shelter_unexpected_detection_frames': SHELTER_UNEXPECTED_DETECTION_FRAMES, 'shelter_correct_zero_rate': SHELTER_CORRECT_ZERO_RATE, 'shelter_unexpected_detection_rate': SHELTER_UNEXPECTED_DETECTION_RATE, 'visible_miss_runs': VISIBLE_MISS_RUNS, 'longest_visible_miss_run_frames': LONGEST_VISIBLE_MISS_RUN_FRAMES, 'longest_visible_miss_run_sec': LONGEST_VISIBLE_MISS_RUN_SEC, 'median_visible_miss_run_frames': MEDIAN_VISIBLE_MISS_RUN_FRAMES, 'p95_visible_miss_run_frames': P95_VISIBLE_MISS_RUN_FRAMES, 'review_images_created': REVIEW_IMAGES_CREATED, 'checkpoint_result': CHECKPOINT_RESULT}
pd.DataFrame([METRICS_ROW]).to_csv(VISIBILITY_METRICS_PATH, index=False)
CONFIG_EVIDENCE = {**CONFIG, 'model_sha256': SOURCE_SUMMARY['model_sha256'], 'video': str(VIDEO_PATH.relative_to(PROJECT_ROOT)), 'video_sha256': VIDEO_SHA256, 'video_fps': VIDEO_FPS, 'total_frames': TOTAL_FRAMES, 'git_commit': GIT_COMMIT}
CONFIG_PATH.write_text(yaml.safe_dump(CONFIG_EVIDENCE, sort_keys=False), encoding='utf-8')
ENVIRONMENT_LINES = [f'experiment_id={EXPERIMENT_ID}', f'datetime_utc={STARTED_AT}', f'git_commit={GIT_COMMIT}', f'python_executable={sys.executable}', f'python={platform.python_version()}', f'conda_env={CONDA_ENV}', f'pandas={pd.__version__}', f'numpy={np.__version__}', f'opencv={cv2.__version__}', f'torch={torch.__version__}', f'cuda_available={CUDA_AVAILABLE}', f'gpu={GPU_NAME}']
ENVIRONMENT_PATH.write_text('\n'.join(ENVIRONMENT_LINES) + '\n', encoding='utf-8')
OUTPUT_FILES = [INTERVALS_PATH, SHORT_ZERO_REVIEW_PATH, FRAME_LABELS_PATH, VISIBILITY_METRICS_PATH, VISIBLE_MISS_RUNS_PATH, CONFIG_PATH, ENVIRONMENT_PATH, SUMMARY_PATH, SHORT_ZERO_REVIEW_DIR, VISIBLE_MISS_REVIEW_DIR, SHELTER_REVIEW_DIR]
SUMMARY = {**METRICS_ROW, 'model_sha256': SOURCE_SUMMARY['model_sha256'], 'video': str(VIDEO_PATH.relative_to(PROJECT_ROOT)), 'video_sha256': VIDEO_SHA256, 'notes': NOTES, 'warnings': WARNINGS, 'output_files': [str(path.relative_to(PROJECT_ROOT)) for path in OUTPUT_FILES], 'git_commit': GIT_COMMIT, 'next_step': 'USER reviews visibility-aware metrics and extracted images, saves Notebook 06a, and reports FINAL SUMMARY; do not start tracking or Notebook 07.'}
SUMMARY_PATH.write_text(json.dumps(SUMMARY, indent=2, ensure_ascii=False), encoding='utf-8')
for note in NOTES: print(f'NOTE: {note}')
for path in OUTPUT_FILES: print(f'Created {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size} bytes)')

## 10. Final Summary

In [ ]:
FINAL_SUMMARY = {'experiment_id': EXPERIMENT_ID, 'total_frames': TOTAL_FRAMES, 'short_zero_runs_total': SHORT_ZERO_RUNS_TOTAL, 'short_zero_runs_reviewed': SHORT_ZERO_RUNS_REVIEWED, 'short_zero_runs_unreviewed': SHORT_ZERO_RUNS_UNREVIEWED, 'visible_frames': VISIBLE_FRAMES, 'in_shelter_frames': IN_SHELTER_FRAMES, 'partially_visible_frames': PARTIALLY_VISIBLE_FRAMES, 'uncertain_frames': UNCERTAIN_FRAMES, 'visible_correct_frames': VISIBLE_CORRECT_FRAMES, 'visible_miss_frames': VISIBLE_MISS_FRAMES, 'visible_overcount_frames': VISIBLE_OVERCOUNT_FRAMES, 'visible_detection_rate': VISIBLE_DETECTION_RATE, 'visible_miss_rate': VISIBLE_MISS_RATE, 'visible_overcount_rate': VISIBLE_OVERCOUNT_RATE, 'shelter_correct_zero_frames': SHELTER_CORRECT_ZERO_FRAMES, 'shelter_unexpected_detection_frames': SHELTER_UNEXPECTED_DETECTION_FRAMES, 'shelter_correct_zero_rate': SHELTER_CORRECT_ZERO_RATE, 'shelter_unexpected_detection_rate': SHELTER_UNEXPECTED_DETECTION_RATE, 'visible_miss_runs': VISIBLE_MISS_RUNS, 'longest_visible_miss_run_frames': LONGEST_VISIBLE_MISS_RUN_FRAMES, 'longest_visible_miss_run_sec': LONGEST_VISIBLE_MISS_RUN_SEC, 'review_images_created': REVIEW_IMAGES_CREATED, 'checkpoint_result': CHECKPOINT_RESULT, 'warnings': WARNINGS, 'output_files': [str(path.relative_to(PROJECT_ROOT)) for path in OUTPUT_FILES], 'next_step': SUMMARY['next_step']}
print('FINAL SUMMARY')
for key, value in FINAL_SUMMARY.items(): print(f'{key}: {value}')